# Q-OPT v2 — IBM Quantum hardware run (budget-guarded)

This notebook executes the pre-registered hardware experiment in
`results/hardware/plan/` (4 instances × {W-state control, XY-QAOA p=1, XY-QAOA p=2,
penalty QAOA p=1} = 16 circuits) as **one** Sampler job, after two short pilot jobs
that measure the real billed QPU time on the chosen backend.

**Budget logic.** Set `BUDGET_SECONDS` to the QPU time you are willing to spend.
The pilots cost a few seconds; the main job's shots per circuit are then chosen
so that the *extrapolated* usage stays below 75 % of the remaining budget.
Nothing is submitted without an explicit `CONFIRM_SUBMIT = True`.

**Rehearsal.** With `USE_FAKE_BACKEND = True` the identical code path runs locally
on `FakeMarrakesh` (Aer, free). Run it that way once before spending QPU time.

All raw counts, job ids, layouts and the calibration snapshot are written to
`results/hardware/run_<backend>_<timestamp>/raw.json` before any analysis.

In [1]:
# ---- user settings --------------------------------------------------------
USE_FAKE_BACKEND = False       # True: free local rehearsal; False: real IBM QPU
BACKEND_NAME     = None        # e.g. "ibm_marrakesh"; None = least busy Heron-class device
BUDGET_SECONDS   = 420         # QPU seconds you allow this notebook to spend (you have ~480)
SHOTS_REQUESTED  = 8192        # per circuit, before budget scaling
CONFIRM_SUBMIT   = True        # must be True to submit the MAIN job to real hardware
PLAN_DIR         = "../results/hardware/plan"

In [2]:
import json, time, warnings, hashlib, platform
from pathlib import Path
import numpy as np
warnings.filterwarnings("ignore")

import qiskit, qiskit_ibm_runtime
from qiskit_ibm_runtime import QiskitRuntimeService, SamplerV2 as Sampler
from qopt.quantum import hardware as hw
from qopt.quantum.train import bootstrap_metrics, repair_counts, metrics_from_feasible_dist
print("qiskit", qiskit.__version__, "| qiskit-ibm-runtime", qiskit_ibm_runtime.__version__,
      "| python", platform.python_version())

qiskit 2.5.2 | qiskit-ibm-runtime 0.49.0 | python 3.10.20


### One-time credentials
Run the cell below **once** with your token and CRN, then set `SAVE_ACCOUNT = False`
and delete the token from the cell. Never commit a notebook that contains a token.

In [ ]:
SAVE_ACCOUNT = False
if SAVE_ACCOUNT:
    QiskitRuntimeService.save_account(
        token="my_api_key",
        instance="my_crn_number",
        overwrite=True,
        set_as_default=True,
    )
if not USE_FAKE_BACKEND:
    service = QiskitRuntimeService()
    backends = service.backends()
    print(f"Account OK. {len(backends)} backend(s) available:")
    for b in backends[:8]:
        print(f"  {b.name} ({b.num_qubits} qubits)")
    try:
        print("Plan usage:", service.usage())
    except Exception as exc:
        print("usage() not available on this plan:", exc)

In [4]:
plan, circuits = hw.load_plan(PLAN_DIR)
plan_hash = hashlib.sha256(Path(PLAN_DIR, "plan.json").read_bytes()).hexdigest()[:12]
print(f"plan {plan_hash}: {len(circuits)} circuits, created {plan['created']}")
for pb in plan["pubs"]:
    print(f"  [{pb['index']:2d}] {pb['instance']:7s} {pb['ansatz']:3s} p={pb['p']}  "
          f"ideal r_feas={pb['ideal']['r_feas']:.3f}  p_feas={pb['ideal']['p_feas']:.3f}")

plan 0b9be7410b19: 16 circuits, created 2026-11-22T05:05:45
  [ 0] R3_s0   W   p=0  ideal r_feas=0.592  p_feas=1.000
  [ 1] R3_s0   XY  p=1  ideal r_feas=0.984  p_feas=1.000
  [ 2] R3_s0   XY  p=2  ideal r_feas=0.990  p_feas=1.000
  [ 3] R3_s0   PEN p=1  ideal r_feas=0.687  p_feas=0.281
  [ 4] R3_s1   W   p=0  ideal r_feas=0.646  p_feas=1.000
  [ 5] R3_s1   XY  p=1  ideal r_feas=0.934  p_feas=1.000
  [ 6] R3_s1   XY  p=2  ideal r_feas=0.971  p_feas=1.000
  [ 7] R3_s1   PEN p=1  ideal r_feas=0.730  p_feas=0.342
  [ 8] R4_s2   W   p=0  ideal r_feas=0.656  p_feas=1.000
  [ 9] R4_s2   XY  p=1  ideal r_feas=0.903  p_feas=1.000
  [10] R4_s2   XY  p=2  ideal r_feas=0.974  p_feas=1.000
  [11] R4_s2   PEN p=1  ideal r_feas=0.721  p_feas=0.258
  [12] R4_s11  W   p=0  ideal r_feas=0.647  p_feas=1.000
  [13] R4_s11  XY  p=1  ideal r_feas=0.900  p_feas=1.000
  [14] R4_s11  XY  p=2  ideal r_feas=0.961  p_feas=1.000
  [15] R4_s11  PEN p=1  ideal r_feas=0.721  p_feas=0.258


In [5]:
if USE_FAKE_BACKEND:
    from qiskit_ibm_runtime.fake_provider import FakeMarrakesh
    backend = FakeMarrakesh()
else:
    if BACKEND_NAME:
        backend = service.backend(BACKEND_NAME)
    else:
        backend = service.least_busy(operational=True, simulator=False, min_num_qubits=127)
    st = backend.status()
    print(f"{backend.name}: operational={st.operational} pending_jobs={st.pending_jobs}")
print("Backend:", backend.name, backend.num_qubits, "qubits")

t0 = time.time()
isa, tstats = hw.transpile_best(circuits, backend, n_seeds=16)
print(f"transpiled in {time.time()-t0:.1f}s")
for s in tstats:
    print(f"  {s['name']:18s} 2q={s['twoq']:4d}  2q-depth={s['twoq_depth']:4d}  layout={s['layout']}")
calib = hw.calibration_snapshot(backend, [s["layout"] for s in tstats])
print(json.dumps({k: v for k, v in calib.items() if k != "physical_qubits"}, indent=1))

ibm_fez: operational=True pending_jobs=0
Backend: ibm_fez 156 qubits
transpiled in 25.1s
  R3_s0|W|p0         2q=  12  2q-depth=   4  layout=[22, 23, 24, 144, 143, 142, 58, 71, 70]
  R3_s0|XY|p1        2q=  51  2q-depth=  19  layout=[121, 122, 123, 58, 71, 70, 144, 143, 136]
  R3_s0|XY|p2        2q= 106  2q-depth=  67  layout=[112, 111, 98, 142, 143, 144, 118, 109, 110]
  R3_s0|PEN|p1       2q=  48  2q-depth=  30  layout=[118, 109, 98, 70, 58, 71, 112, 110, 111]
  R3_s1|W|p0         2q=  12  2q-depth=   4  layout=[22, 23, 24, 144, 143, 142, 58, 71, 70]
  R3_s1|XY|p1        2q=  51  2q-depth=  19  layout=[144, 143, 136, 121, 122, 123, 58, 71, 70]
  R3_s1|XY|p2        2q= 106  2q-depth=  66  layout=[112, 111, 98, 118, 109, 110, 142, 143, 144]
  R3_s1|PEN|p1       2q=  50  2q-depth=  30  layout=[118, 109, 98, 112, 110, 111, 70, 58, 71]
  R4_s2|W|p0         2q=  16  2q-depth=   4  layout=[2, 3, 4, 58, 71, 70, 24, 23, 22, 142, 143, 144]
  R4_s2|XY|p1        2q=  77  2q-depth=  34  layout=[8

### Pilot jobs (≈ 4–8 QPU seconds)
Two single-circuit jobs with different shot counts give a linear model of billed
seconds = overhead + rate × shots on *this* backend today.

In [6]:
sampler = Sampler(mode=backend)
applied = hw.configure_sampler(sampler, dd=True, twirl=True)
print("error suppression:", applied)

pilot_idx = int(np.argmax([s["twoq"] for s in tstats]))   # deepest circuit = worst case
pilot_points, pilot_jobs = [], []
for shots in (512, 4096):
    job = sampler.run([isa[pilot_idx]], shots=shots)
    res = job.result()
    qs = hw.quantum_seconds(job)
    pilot_points.append((shots, qs))
    pilot_jobs.append(dict(job_id=job.job_id(), shots=shots, quantum_seconds=qs))
    print(f"pilot {shots:5d} shots -> job {job.job_id()}  billed={qs}")
per_shot, overhead, how = hw.fit_usage(pilot_points)
spent = sum(q for _, q in pilot_points if q) if not USE_FAKE_BACKEND else 0.0
remaining = BUDGET_SECONDS - spent
shots = hw.plan_shots(per_shot, overhead, len(isa), SHOTS_REQUESTED, remaining)
pred = overhead + per_shot * shots * len(isa)
print(f"usage model ({how}): overhead={overhead:.2f}s  rate={per_shot*1e3:.3f} ms/shot")
print(f"pilots spent {spent:.1f}s; remaining budget {remaining:.1f}s")
print(f"main job: {len(isa)} circuits x {shots} shots  ->  predicted {pred:.1f}s")
assert shots >= 1024, "Budget too small for a meaningful main job; stop here."

error suppression: {'dynamical_decoupling': 'XpXm', 'twirling': 'gates+measure, auto'}
pilot   512 shots -> job dc1d9n6q4uuc73a3pgo0  billed=0.0
pilot  4096 shots -> job dc1d9pinj2vs73df6slg  billed=0.0
usage model (single-point): overhead=4.00s  rate=0.600 ms/shot
pilots spent 0.0s; remaining budget 420.0s
main job: 16 circuits x 8192 shots  ->  predicted 82.6s


In [7]:
if not USE_FAKE_BACKEND and not CONFIRM_SUBMIT:
    raise SystemExit("Review the prediction above, then set CONFIRM_SUBMIT = True and re-run this cell.")
t_submit = time.time()
job = sampler.run(isa, shots=shots)
print("main job submitted:", job.job_id())
result = job.result()
qs_main = hw.quantum_seconds(job)
print(f"done in {time.time()-t_submit:.0f}s wall; billed QPU seconds = {qs_main}")

main job submitted: dc1d9seq4uuc73a3ph20
done in 63s wall; billed QPU seconds = 38.0


In [8]:
counts = [hw.counts_from_pub_result(r) for r in result]
stamp = time.strftime("%Y%m%d-%H%M%S")
tag = "fake" if USE_FAKE_BACKEND else "qpu"
run_dir = Path("../results/hardware") / f"run_{tag}_{backend.name}_{stamp}"
run_dir.mkdir(parents=True, exist_ok=True)
raw = dict(plan_hash=plan_hash, backend=backend.name, fake_backend=USE_FAKE_BACKEND,
           submitted=time.strftime("%Y-%m-%dT%H:%M:%S", time.localtime(t_submit)),
           job_id=job.job_id(), shots_per_circuit=shots, billed_quantum_seconds=qs_main,
           pilot_jobs=pilot_jobs, usage_model=dict(per_shot_s=per_shot, overhead_s=overhead, method=how),
           error_suppression=applied, calibration=calib, transpile=tstats,
           versions=dict(qiskit=qiskit.__version__, runtime=qiskit_ibm_runtime.__version__),
           pubs=[dict(**pb, counts=c) for pb, c in zip(plan["pubs"], counts)])
(run_dir / "raw.json").write_text(json.dumps(raw, indent=1))
print("saved", run_dir / "raw.json")

saved ../results/hardware/run_qpu_ibm_fez_20261122-114238/raw.json


### Quick look (full analysis: `python experiments/exp07_hardware_analysis.py --run-dir <run_dir>`)

In [9]:
from qopt.quantum.hardware import problem_from_plan
inst = {i["name"]: i for i in plan["instances"]}
print(f"{'circuit':18s} {'p_feas':>14s} {'r_feas (post-sel.)':>22s} {'ideal':>7s} {'random':>7s}")
for pb, c in zip(plan["pubs"], counts):
    I = inst[pb["instance"]]; prob = problem_from_plan(I); E = np.array(I["spectrum"])
    b = bootstrap_metrics(prob, c, E, n_boot=300)
    pt, ci = b["point"], b["ci"]
    print(f"{pb['instance']+'|'+pb['ansatz']+str(pb['p']):18s} "
          f"{pt['p_feas']:.3f} [{ci['p_feas'][0]:.3f},{ci['p_feas'][1]:.3f}] "
          f"{pt['r_feas']:.3f} [{ci['r_feas'][0]:.3f},{ci['r_feas'][1]:.3f}]   "
          f"{pb['ideal']['r_feas']:.3f}  {pt['r_rand']:.3f}")

circuit                    p_feas     r_feas (post-sel.)   ideal  random
R3_s0|W0           0.840 [0.831,0.847] 0.587 [0.577,0.597]   0.592  0.592
R3_s0|XY1          0.652 [0.642,0.663] 0.966 [0.963,0.969]   0.984  0.592
R3_s0|XY2          0.419 [0.408,0.431] 0.922 [0.914,0.931]   0.990  0.592
R3_s0|PEN1         0.210 [0.202,0.219] 0.668 [0.650,0.688]   0.687  0.592
R3_s1|W0           0.839 [0.830,0.846] 0.642 [0.633,0.651]   0.646  0.646
R3_s1|XY1          0.661 [0.650,0.670] 0.918 [0.915,0.921]   0.934  0.646
R3_s1|XY2          0.402 [0.393,0.413] 0.920 [0.913,0.926]   0.971  0.646
R3_s1|PEN1         0.255 [0.246,0.264] 0.728 [0.711,0.743]   0.730  0.646
R4_s2|W0           0.774 [0.765,0.784] 0.644 [0.637,0.652]   0.656  0.656
R4_s2|XY1          0.460 [0.449,0.471] 0.887 [0.882,0.891]   0.903  0.656
R4_s2|XY2          0.223 [0.214,0.232] 0.903 [0.895,0.911]   0.974  0.656
R4_s2|PEN1         0.146 [0.139,0.154] 0.705 [0.686,0.722]   0.721  0.656
R4_s11|W0          0.784 [0.775,0.793] 